In [13]:
import os
import re
import sys
import string
import pandas as pd

sys.path.append(os.path.abspath('../scripts'))

from kworb import Kworb
from genius import Genius

genius = Genius()
kworb = Kworb()

In [14]:
def dom_to_plain_text(dom: dict) -> str:
    if isinstance(dom, str):
        return dom.strip()
    if 'children' in dom:
        return ' '.join([dom_to_plain_text(child_dom) for child_dom in dom['children']])
    if len(dom) == 1 or dom['tag'] in {'img', 'iframe', 'a'}:
        return ' '
    print(dom)
    assert 0 == 1

def get_description(item: dict) -> str:
    description = dom_to_plain_text(item['description']['dom'])
    description = re.sub(rf'\\s+([{re.escape(string.punctuation)}])', r'\\1', description)
    description = re.sub(r'\\s+', ' ', description)
    return description

In [15]:
df = kworb.spotify_daily_chart_totals('2026-04-09')

**Поиск песен на Genius**

In [16]:
def get_genius_ids(row):
    query = f'{row.track_title} — {row.artist_name}'
    response = Genius().search(query)
    if response['response']['hits']:
        result = response['response']['hits'][0]['result']
        return pd.Series([result['id'], result['primary_artist']['id']])
    return pd.Series([None, None])

df[['genius_song_id', 'genius_artist_id']] = df.apply(get_genius_ids, axis=1)

**Получение признаков песен с Genius**

In [17]:
def get_youtube_url(song: dict) -> str:
    for media in song['media']:
        if media['provider'] == 'youtube':
            return media['url']

def get_song_features(song_id):
    if not pd.isna(song_id):
        song = genius.song(int(song_id))['response']['song']
        return pd.Series({
            'album': song['album']['name'] if song['album'] else None,
            'release_date': song['release_date'],
            'language': song['language'],
            'pageviews': song['stats'].get('pageviews'),
            'feat_artist_count': len(song['featured_artists']),
            'translate_count': len(song['translation_songs']),
            'third_platform_count': len(song['media']),
            'has_sample': len(song['song_relationships'][0]['songs']) > 0,
            'is_cover': len(song['song_relationships'][4]['songs']) > 0,
            'is_remix': len(song['song_relationships'][6]['songs']) > 0,
            'is_translation': len(song['song_relationships'][-2]['songs']) > 0,
            'image_url': song['header_image_url'],
            'primary_color': song['song_art_primary_color'],
            'secondary_color': song['song_art_secondary_color'],
            'text_color': song['song_art_text_color'],
            'annotation_count': song['annotation_count'],
            'song_description': get_description(song),
            'youtube_url': get_youtube_url(song),
            'genius_song_url': song['url'],
        })
    return pd.Series({
            'album': None,
            'release_date': None,
            'language': None,
            'pageviews': None,
            'feat_artist_count': None,
            'translate_count': None,
            'third_platform_count': None,
            'has_sample': None,
            'is_cover': None,
            'is_remix': None,
            'is_translation': None,
            'image_url': None,
            'primary_color': None,
            'secondary_color': None,
            'text_color': None,
            'annotation_count': None,
            'song_description': None,
            'youtube_url': None,
            'genius_song_url': None,
        })

song_features = df['genius_song_id'].apply(get_song_features)

df = df.join(song_features)

**Получение признаков артистов с Genius**

In [18]:
def get_artist_features(artist_id):
    if not pd.isna(artist_id):
        artist = genius.artist(int(artist_id))['response']['artist']
        return pd.Series({
            'artist_description': get_description(artist),
        })
    return pd.Series({
            'artist_description': None,
        })

artist_features = df['genius_artist_id'].apply(get_artist_features)

df = df.join(artist_features)

**Парсинг текстов песен с Genius**

In [20]:
df['lyrics'] = df['genius_song_url'].apply(lambda x: genius.lyrics(x) if pd.notna(x) else None)

---
---
---

In [21]:
df.to_csv('../data/selected.csv')